# 🚕 RideConnect — Full ML Pipeline
### From Raw Kigali Rides Data → Trained Models → FastAPI Service → Supabase Integration

**What this notebook builds:**
- ✅ Dataset engineering from your real `kigali_rides.csv`
- ✅ Model 1: Demand Forecasting (LSTM)
- ✅ Model 2: Intelligent Ride Matching (Random Forest)
- ✅ Model 3: Driver Behavior Classification (Gradient Boosting)
- ✅ Model 4: Surge Price Predictor (XGBoost)
- ✅ Supabase live data connector (fetches from your Laravel DB)
- ✅ Nightly retraining pipeline
- ✅ Export models for FastAPI deployment

---
> **How to use:** Run all cells top to bottom. Upload `kigali_rides.csv` when prompted in Step 1.
> For Supabase live data, fill in your credentials in Step 6.

## ⚙️ STEP 0 — Install Dependencies

In [ ]:
!pip install -q pandas numpy scikit-learn xgboost tensorflow keras joblib
!pip install -q supabase python-dotenv fastapi uvicorn requests matplotlib seaborn
!pip install -q google-colab-drive  # for saving to Google Drive

print('✅ All dependencies installed')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os
import json
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, mean_absolute_error
)
from xgboost import XGBRegressor, XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

warnings.filterwarnings('ignore')
os.makedirs('/content/rideconnect_models', exist_ok=True)
os.makedirs('/content/rideconnect_data',   exist_ok=True)

print('✅ Imports done')
print(f'TensorFlow: {tf.__version__}')

---
## 📂 STEP 1 — Load Your kigali_rides.csv

In [ ]:
from google.colab import files

print('📤 Upload your kigali_rides.csv file:')
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(filename)

print(f'\n✅ Loaded {len(df_raw)} rides with {len(df_raw.columns)} columns')
df_raw.head(3)

In [ ]:
print('=== Dataset Overview ===')
print(f'Total rides   : {len(df_raw)}')
print(f'Completed     : {(df_raw.status == "completed").sum()}')
print(f'Cancelled     : {(df_raw.status == "cancelled").sum()}')
print(f'Unique drivers: {df_raw.driver_id.nunique()}')
print(f'Unique users  : {df_raw.user_id.nunique()}')
print(f'\nNull values:\n{df_raw.isnull().sum()}')

---
## 🔧 STEP 2 — Feature Engineering (Build Your Dataset)

We transform the raw CSV into rich ML features:
- Time features (hour, day, rush hour flags)
- GPS zone mapping (Kigali neighborhoods from coordinates)
- Wait time, demand level, driver reliability score
- Rolling demand per zone-hour window

In [ ]:
df = df_raw.copy()

# ── 1. Parse timestamps ───────────────────────────────────────────────────────
df['request_time']  = pd.to_datetime(df['request_time'],  dayfirst=True)
df['pickup_time']   = pd.to_datetime(df['pickup_time'],   dayfirst=True)
df['dropoff_time']  = pd.to_datetime(df['dropoff_time'],  dayfirst=True)

# ── 2. Time features ──────────────────────────────────────────────────────────
df['hour']          = df['request_time'].dt.hour
df['weekday']       = df['request_time'].dt.dayofweek       # 0=Mon, 6=Sun
df['month']         = df['request_time'].dt.month
df['day_of_month']  = df['request_time'].dt.day
df['is_weekend']    = (df['weekday'] >= 5).astype(int)

# Morning rush: 7-9, Evening rush: 17-19, Night: 21-23, Off-peak otherwise
def get_time_period(h):
    if   7  <= h <= 9:  return 'morning_rush'
    elif 17 <= h <= 19: return 'evening_rush'
    elif 20 <= h <= 23: return 'night'
    elif 0  <= h <= 5:  return 'late_night'
    else:               return 'off_peak'

df['time_period']   = df['hour'].apply(get_time_period)
df['is_rush_hour']  = df['time_period'].isin(['morning_rush','evening_rush']).astype(int)

# ── 3. Wait time (minutes between request and pickup) ────────────────────────
df['wait_time_min'] = (df['pickup_time'] - df['request_time']).dt.total_seconds() / 60
df['wait_time_min'] = df['wait_time_min'].clip(lower=0)

# ── 4. Map GPS coordinates → Kigali Zone ─────────────────────────────────────
# Kigali zones defined by lat/lng bounding boxes
KIGALI_ZONES = {
    'Nyabugogo':  {'lat': (-1.95, -1.98),  'lng': (30.04, 30.08)},
    'CBD':        {'lat': (-1.94, -1.96),  'lng': (30.06, 30.10)},
    'Kacyiru':    {'lat': (-1.92, -1.95),  'lng': (30.09, 30.13)},
    'Remera':     {'lat': (-1.90, -1.93),  'lng': (30.10, 30.14)},
    'Kimironko':  {'lat': (-1.90, -1.93),  'lng': (30.12, 30.15)},
    'Kicukiro':   {'lat': (-1.96, -1.98),  'lng': (30.10, 30.14)},
    'Gisozi':     {'lat': (-1.90, -1.93),  'lng': (30.07, 30.10)},
    'Kanombe':    {'lat': (-1.95, -1.98),  'lng': (30.12, 30.15)},
}

def coords_to_zone(lat, lng):
    for zone, bounds in KIGALI_ZONES.items():
        lat_min, lat_max = min(bounds['lat']), max(bounds['lat'])
        lng_min, lng_max = min(bounds['lng']), max(bounds['lng'])
        if lat_min <= lat <= lat_max and lng_min <= lng <= lng_max:
            return zone
    return 'Other'

df['pickup_zone']  = df.apply(lambda r: coords_to_zone(r['pickup_lat'],  r['pickup_lng']),  axis=1)
df['dropoff_zone'] = df.apply(lambda r: coords_to_zone(r['dropoff_lat'], r['dropoff_lng']), axis=1)

# ── 5. Demand level per zone-hour ─────────────────────────────────────────────
zone_hour_counts = df.groupby(['pickup_zone', 'hour']).size().reset_index(name='zone_hour_count')
df = df.merge(zone_hour_counts, on=['pickup_zone', 'hour'], how='left')

p33 = df['zone_hour_count'].quantile(0.33)
p66 = df['zone_hour_count'].quantile(0.66)

def get_demand_level(c):
    if   c <= p33: return 'low'
    elif c <= p66: return 'medium'
    else:          return 'high'

df['demand_level'] = df['zone_hour_count'].apply(get_demand_level)

# ── 6. Driver reliability score (per driver historical stats) ─────────────────
driver_stats = df.groupby('driver_id').agg(
    driver_cancel_rate  = ('status', lambda x: (x == 'cancelled').mean()),
    driver_avg_rating   = ('driver_rating', 'mean'),
    driver_total_rides  = ('ride_id', 'count'),
    driver_avg_idle     = ('driver_idle_time', 'mean'),
).reset_index()

df = df.merge(driver_stats, on='driver_id', how='left')

# ── 7. Fare per km (pricing efficiency) ───────────────────────────────────────
df['fare_per_km']   = df['fare_rwf'] / df['distance_km'].replace(0, np.nan)
df['fare_per_min']  = df['fare_rwf'] / df['duration_min'].replace(0, np.nan)

# ── 8. Binary target columns ──────────────────────────────────────────────────
df['is_cancelled']  = (df['status'] == 'cancelled').astype(int)
df['is_completed']  = (df['status'] == 'completed').astype(int)
df['is_risky_driver'] = (
    (df['driver_cancel_rate'] > 0.3) | (df['driver_avg_rating'] < 3.5)
).astype(int)

# ── 9. Save engineered dataset ────────────────────────────────────────────────
df.to_csv('/content/rideconnect_data/kigali_rides_engineered.csv', index=False)

print(f'✅ Feature engineering done. Dataset shape: {df.shape}')
print(f'\nNew columns added:')
new_cols = [c for c in df.columns if c not in df_raw.columns]
for c in new_cols:
    print(f'  + {c}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('RideConnect — Kigali Rides EDA', fontsize=16, fontweight='bold')

# Rides by hour
df['hour'].value_counts().sort_index().plot(kind='bar', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Rides by Hour')
axes[0,0].set_xlabel('Hour of Day')

# Status distribution
df['status'].value_counts().plot(kind='pie', ax=axes[0,1], autopct='%1.1f%%',
                                  colors=['#2ecc71','#e74c3c'])
axes[0,1].set_title('Ride Status')

# Pickup zone distribution
df['pickup_zone'].value_counts().plot(kind='barh', ax=axes[0,2], color='coral')
axes[0,2].set_title('Pickup Zones')

# Demand level
df['demand_level'].value_counts().plot(kind='bar', ax=axes[1,0], color='purple', alpha=0.7)
axes[1,0].set_title('Demand Level Distribution')

# Fare vs Distance
completed = df[df['status'] == 'completed']
axes[1,1].scatter(completed['distance_km'], completed['fare_rwf'], alpha=0.4, color='teal')
axes[1,1].set_title('Fare vs Distance')
axes[1,1].set_xlabel('Distance (km)')
axes[1,1].set_ylabel('Fare (RWF)')

# Driver rating distribution
df['driver_rating'].hist(ax=axes[1,2], bins=15, color='gold', edgecolor='black')
axes[1,2].set_title('Driver Rating Distribution')

plt.tight_layout()
plt.savefig('/content/rideconnect_data/eda_plots.png', dpi=120)
plt.show()
print('✅ EDA plots saved')

---
## 🏗️ STEP 3 — Encode & Prepare All Features

In [ ]:
# ── Label Encoders ────────────────────────────────────────────────────────────
le_zone        = LabelEncoder()
le_period      = LabelEncoder()
le_demand      = LabelEncoder()

df['pickup_zone_enc']  = le_zone.fit_transform(df['pickup_zone'])
df['dropoff_zone_enc'] = le_zone.transform(
    df['dropoff_zone'].map(lambda z: z if z in le_zone.classes_ else 'Other')
)
df['time_period_enc']  = le_period.fit_transform(df['time_period'])
df['demand_level_enc'] = le_demand.fit_transform(df['demand_level'])

# ── Save encoders ─────────────────────────────────────────────────────────────
joblib.dump(le_zone,   '/content/rideconnect_models/le_zone.pkl')
joblib.dump(le_period, '/content/rideconnect_models/le_period.pkl')
joblib.dump(le_demand, '/content/rideconnect_models/le_demand.pkl')

# ── Zone mapping reference (for FastAPI) ─────────────────────────────────────
zone_mapping = {zone: int(enc) for zone, enc in
                zip(le_zone.classes_, le_zone.transform(le_zone.classes_))}
with open('/content/rideconnect_models/zone_mapping.json', 'w') as f:
    json.dump(zone_mapping, f, indent=2)

print('✅ Encoders created and saved')
print('Zone mapping:', zone_mapping)

---
## 🧠 STEP 4A — Model 1: Demand Forecasting (LSTM)

**Predicts:** `demand_level` (low / medium / high) per zone + time

**Used in Laravel:** Before a passenger even opens the app — drivers are pre-positioned

In [ ]:
# ── Features for demand prediction ───────────────────────────────────────────
DEMAND_FEATURES = [
    'hour', 'weekday', 'month', 'is_weekend', 'is_rush_hour',
    'pickup_zone_enc', 'time_period_enc', 'zone_hour_count'
]
DEMAND_TARGET = 'demand_level_enc'  # 0=high, 1=low, 2=medium

X_demand = df[DEMAND_FEATURES].values
y_demand = df[DEMAND_TARGET].values

# Scale features
scaler_demand = MinMaxScaler()
X_scaled = scaler_demand.fit_transform(X_demand)
joblib.dump(scaler_demand, '/content/rideconnect_models/scaler_demand.pkl')

# Reshape for LSTM: [samples, timesteps=1, features]
X_lstm = X_scaled.reshape((X_scaled.shape[0], 1, X_scaled.shape[1]))

# Train/test split (keep time order)
split = int(0.8 * len(X_lstm))
X_train, X_test = X_lstm[:split], X_lstm[split:]
y_train, y_test = y_demand[:split], y_demand[split:]

n_classes = len(np.unique(y_demand))
n_features = len(DEMAND_FEATURES)

# ── Build LSTM model ──────────────────────────────────────────────────────────
demand_model = Sequential([
    LSTM(128, input_shape=(1, n_features), return_sequences=True),
    Dropout(0.3),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    BatchNormalization(),
    Dense(32, activation='relu'),
    Dense(n_classes, activation='softmax')
])

demand_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
demand_model.summary()

# ── Train ─────────────────────────────────────────────────────────────────────
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint(
    '/content/rideconnect_models/demand_lstm_best.h5',
    monitor='val_accuracy', save_best_only=True
)

history = demand_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.15,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

# ── Evaluate ──────────────────────────────────────────────────────────────────
loss, acc = demand_model.evaluate(X_test, y_test, verbose=0)
y_pred_demand = np.argmax(demand_model.predict(X_test), axis=1)

print(f'\n✅ Demand LSTM — Test Accuracy: {acc:.2%}')
print(classification_report(
    y_test, y_pred_demand,
    target_names=le_demand.classes_
))

demand_model.save('/content/rideconnect_models/demand_lstm.h5')
print('✅ Demand model saved: demand_lstm.h5')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(history.history['accuracy'],    label='Train Acc')
ax1.plot(history.history['val_accuracy'],label='Val Acc')
ax1.set_title('Demand LSTM — Accuracy')
ax1.legend()

ax2.plot(history.history['loss'],    label='Train Loss')
ax2.plot(history.history['val_loss'],label='Val Loss')
ax2.set_title('Demand LSTM — Loss')
ax2.legend()

plt.tight_layout()
plt.savefig('/content/rideconnect_data/demand_lstm_training.png', dpi=100)
plt.show()

---
## 🤝 STEP 4B — Model 2: Intelligent Ride Matching (Random Forest)

**Predicts:** probability that a driver-passenger match will `complete` (not cancel)

**Used in Laravel:** When a ride is requested — score all nearby drivers, pick the best

In [ ]:
MATCH_FEATURES = [
    'hour', 'weekday', 'is_weekend', 'is_rush_hour',
    'pickup_zone_enc', 'dropoff_zone_enc',
    'distance_km', 'driver_rating',
    'driver_idle_time', 'driver_cancel_rate',
    'driver_avg_rating', 'driver_total_rides',
    'surge_multiplier', 'demand_level_enc'
]
MATCH_TARGET = 'is_completed'

df_match = df[MATCH_FEATURES + [MATCH_TARGET]].dropna()

X_match = df_match[MATCH_FEATURES]
y_match = df_match[MATCH_TARGET]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_match, y_match, test_size=0.2, random_state=42, stratify=y_match
)

# ── Random Forest ─────────────────────────────────────────────────────────────
matching_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
matching_model.fit(X_tr, y_tr)

y_pred_match = matching_model.predict(X_te)
match_acc = accuracy_score(y_te, y_pred_match)

print(f'\n✅ Matching Model — Test Accuracy: {match_acc:.2%}')
print(classification_report(y_te, y_pred_match, target_names=['cancelled','completed']))

joblib.dump(matching_model, '/content/rideconnect_models/matching_rf.pkl')
joblib.dump(MATCH_FEATURES, '/content/rideconnect_models/matching_features.pkl')
print('✅ Matching model saved: matching_rf.pkl')

In [ ]:
# Feature importance plot
importances = pd.Series(
    matching_model.feature_importances_,
    index=MATCH_FEATURES
).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('Ride Matching — Feature Importance')
plt.tight_layout()
plt.savefig('/content/rideconnect_data/matching_importance.png', dpi=100)
plt.show()

---
## 🚨 STEP 4C — Model 3: Driver Behavior Classification (Gradient Boosting)

**Predicts:** Is this driver likely to be risky (cancel, low rating, long idle)?

**Used in Laravel:** Admin dashboard alerts + exclude risky drivers from matching

In [ ]:
BEHAVIOR_FEATURES = [
    'hour', 'weekday', 'is_rush_hour',
    'driver_rating', 'driver_idle_time',
    'driver_total_rides', 'driver_cancel_rate',
    'distance_km', 'duration_min',
    'fare_rwf', 'surge_multiplier',
    'pickup_zone_enc'
]
BEHAVIOR_TARGET = 'is_risky_driver'

df_beh = df[BEHAVIOR_FEATURES + [BEHAVIOR_TARGET]].dropna()
X_beh = df_beh[BEHAVIOR_FEATURES]
y_beh = df_beh[BEHAVIOR_TARGET]

X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_beh, y_beh, test_size=0.2, random_state=42
)

behavior_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)
behavior_model.fit(X_tr_b, y_tr_b)

beh_acc = behavior_model.score(X_te_b, y_te_b)
y_pred_beh = behavior_model.predict(X_te_b)

print(f'\n✅ Behavior Model — Test Accuracy: {beh_acc:.2%}')
print(classification_report(y_te_b, y_pred_beh, target_names=['reliable','risky']))

joblib.dump(behavior_model,   '/content/rideconnect_models/behavior_gb.pkl')
joblib.dump(BEHAVIOR_FEATURES,'/content/rideconnect_models/behavior_features.pkl')
print('✅ Behavior model saved: behavior_gb.pkl')

---
## 💰 STEP 4D — Model 4: Surge Price Predictor (XGBoost)

**Predicts:** `surge_multiplier` (1.0 → 2.5) based on demand and time conditions

**Used in Laravel:** Show fair predicted fare to passenger before booking

In [ ]:
SURGE_FEATURES = [
    'hour', 'weekday', 'is_weekend', 'is_rush_hour', 'month',
    'pickup_zone_enc', 'distance_km',
    'zone_hour_count', 'demand_level_enc',
    'driver_idle_time', 'wait_time_min'
]
SURGE_TARGET = 'surge_multiplier'

df_surge = df[SURGE_FEATURES + [SURGE_TARGET]].dropna()
X_surge = df_surge[SURGE_FEATURES]
y_surge = df_surge[SURGE_TARGET]

X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X_surge, y_surge, test_size=0.2, random_state=42
)

surge_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
surge_model.fit(X_tr_s, y_tr_s,
                eval_set=[(X_te_s, y_te_s)],
                verbose=False)

y_pred_surge = surge_model.predict(X_te_s)
mae = mean_absolute_error(y_te_s, y_pred_surge)

print(f'\n✅ Surge Model — Mean Absolute Error: {mae:.4f}')
print(f'   Average actual surge: {y_te_s.mean():.3f}')
print(f'   Average predicted   : {y_pred_surge.mean():.3f}')

joblib.dump(surge_model,   '/content/rideconnect_models/surge_xgb.pkl')
joblib.dump(SURGE_FEATURES,'/content/rideconnect_models/surge_features.pkl')
print('✅ Surge model saved: surge_xgb.pkl')

---
## 📊 STEP 5 — Model Summary Report

In [ ]:
print('=' * 60)
print('    RIDECONNECT MODEL TRAINING SUMMARY')
print('=' * 60)
print(f"  Dataset         : {len(df)} rides (Kigali 2024)")
print(f"  Training date   : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print()
print('  MODEL 1 — Demand Forecasting (LSTM)')
print(f"    Accuracy        : {acc:.2%}")
print(f"    Output          : low / medium / high demand")
print(f"    File            : demand_lstm.h5")
print()
print('  MODEL 2 — Ride Matching (Random Forest)')
print(f"    Accuracy        : {match_acc:.2%}")
print(f"    Output          : completion probability [0-1]")
print(f"    File            : matching_rf.pkl")
print()
print('  MODEL 3 — Driver Behavior (Gradient Boosting)')
print(f"    Accuracy        : {beh_acc:.2%}")
print(f"    Output          : reliable / risky")
print(f"    File            : behavior_gb.pkl")
print()
print('  MODEL 4 — Surge Pricing (XGBoost)')
print(f"    MAE             : {mae:.4f}")
print(f"    Output          : surge multiplier (1.0–2.5)")
print(f"    File            : surge_xgb.pkl")
print('=' * 60)

---
## 🔌 STEP 6 — Supabase Live Data Connector

**This pulls real rides from your Supabase DB (connected to your Laravel backend)**

Fill in your Supabase credentials below — find them at:
`Supabase Dashboard → Settings → API`

In [ ]:
# ── FILL THESE IN ─────────────────────────────────────────────────────────────
SUPABASE_URL     = 'https://YOUR_PROJECT_ID.supabase.co'   # e.g. https://abcxyz.supabase.co
SUPABASE_KEY     = 'YOUR_SERVICE_ROLE_KEY'                  # Settings → API → service_role
SUPABASE_TABLE   = 'historical_data'                        # your table name in Supabase
# ─────────────────────────────────────────────────────────────────────────────

def fetch_from_supabase(limit=5000, offset=0):
    """
    Fetches rides from Supabase using REST API.
    Returns a pandas DataFrame.
    Maps your DB column names to the ML pipeline feature names.
    """
    import requests

    url = f"{SUPABASE_URL}/rest/v1/{SUPABASE_TABLE}"
    headers = {
        'apikey': SUPABASE_KEY,
        'Authorization': f'Bearer {SUPABASE_KEY}',
        'Range': f'{offset}-{offset + limit - 1}'
    }
    params = {'select': '*', 'order': 'created_at.desc'}

    resp = requests.get(url, headers=headers, params=params)

    if resp.status_code != 200:
        print(f'❌ Supabase error {resp.status_code}: {resp.text}')
        return None

    data = resp.json()
    df_live = pd.DataFrame(data)

    # ── Column mapping: DB column → ML feature name ──────────────────────────
    # Adjust this mapping to match your actual Supabase table columns!
    column_map = {
        'ride_id':          'ride_id',
        'timestamp':        'request_time',
        'traffic_condition':'traffic_condition',
        'demand_level':     'demand_level',
        # Add more mappings from your historical_data table schema
    }
    df_live = df_live.rename(columns=column_map)

    print(f'✅ Fetched {len(df_live)} records from Supabase')
    return df_live


def save_live_data_for_retraining(df_live):
    """Appends new Supabase data to the local training dataset."""
    existing_path = '/content/rideconnect_data/kigali_rides_engineered.csv'
    df_existing = pd.read_csv(existing_path)

    combined = pd.concat([df_existing, df_live], ignore_index=True)
    combined = combined.drop_duplicates(subset=['ride_id'])
    combined.to_csv(existing_path, index=False)

    print(f'✅ Dataset updated: {len(df_existing)} → {len(combined)} rows')
    return combined


# ── Test connection ────────────────────────────────────────────────────────────
print('Testing Supabase connection...')
print('(Fill in SUPABASE_URL and SUPABASE_KEY above before running)')
# df_live = fetch_from_supabase()  # Uncomment when credentials are filled

---
## 🔄 STEP 7 — Nightly Retraining Pipeline

This pipeline:
1. Fetches new rides from Supabase
2. Merges with existing training data
3. Re-trains all 4 models
4. Saves updated models (overwrites old ones)

**Run this cell manually every night, OR set it as a Colab scheduled job**

In [ ]:
def full_retrain_pipeline(df_training):
    """
    Complete retrain of all 4 models on the given DataFrame.
    df_training must have all engineered features already.
    """
    print(f'🔄 Starting full retrain on {len(df_training)} records...')
    start = datetime.now()

    # Re-encode
    le_z = joblib.load('/content/rideconnect_models/le_zone.pkl')
    le_d = joblib.load('/content/rideconnect_models/le_demand.pkl')
    le_p = joblib.load('/content/rideconnect_models/le_period.pkl')

    df_t = df_training.copy()
    df_t['pickup_zone_enc']  = df_t['pickup_zone'].map(lambda z:
        le_z.transform([z])[0] if z in le_z.classes_ else -1)
    df_t['demand_level_enc'] = df_t['demand_level'].map(lambda d:
        le_d.transform([d])[0] if d in le_d.classes_ else 1)
    df_t['time_period_enc']  = df_t['time_period'].map(lambda p:
        le_p.transform([p])[0] if p in le_p.classes_ else 0)

    results = {}

    # ── Model 2: Matching ──────────────────────────────────────────────────────
    mf = joblib.load('/content/rideconnect_models/matching_features.pkl')
    dm = df_t[mf + ['is_completed']].dropna()
    Xm, ym = dm[mf], dm['is_completed']
    Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=0.2, random_state=42)
    m = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
    m.fit(Xtr, ytr)
    results['matching'] = m.score(Xte, yte)
    joblib.dump(m, '/content/rideconnect_models/matching_rf.pkl')

    # ── Model 3: Behavior ──────────────────────────────────────────────────────
    bf = joblib.load('/content/rideconnect_models/behavior_features.pkl')
    db = df_t[bf + ['is_risky_driver']].dropna()
    Xb, yb = db[bf], db['is_risky_driver']
    Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.2, random_state=42)
    b = GradientBoostingClassifier(n_estimators=150, random_state=42)
    b.fit(Xtr, ytr)
    results['behavior'] = b.score(Xte, yte)
    joblib.dump(b, '/content/rideconnect_models/behavior_gb.pkl')

    # ── Model 4: Surge ─────────────────────────────────────────────────────────
    sf = joblib.load('/content/rideconnect_models/surge_features.pkl')
    ds = df_t[sf + ['surge_multiplier']].dropna()
    Xs, ys = ds[sf], ds['surge_multiplier']
    Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=0.2, random_state=42)
    s = XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
    s.fit(Xtr, ytr)
    results['surge_mae'] = mean_absolute_error(yte, s.predict(Xte))
    joblib.dump(s, '/content/rideconnect_models/surge_xgb.pkl')

    # ── Log retrain results ────────────────────────────────────────────────────
    duration = (datetime.now() - start).seconds
    log = {
        'retrain_time': datetime.now().isoformat(),
        'total_records': len(df_training),
        'matching_accuracy': round(results['matching'], 4),
        'behavior_accuracy': round(results['behavior'], 4),
        'surge_mae':         round(results['surge_mae'], 4),
        'duration_seconds':  duration
    }
    with open('/content/rideconnect_models/retrain_log.json', 'w') as f:
        json.dump(log, f, indent=2)

    print(f'\n✅ Retrain complete in {duration}s')
    print(f"   Matching accuracy  : {results['matching']:.2%}")
    print(f"   Behavior accuracy  : {results['behavior']:.2%}")
    print(f"   Surge MAE          : {results['surge_mae']:.4f}")
    return log


def nightly_retrain_with_supabase():
    """Full pipeline: fetch new data → merge → retrain all models."""
    print('=== NIGHTLY RETRAIN PIPELINE ===')

    # 1. Fetch new data from Supabase
    df_live = fetch_from_supabase(limit=5000)
    if df_live is None or df_live.empty:
        print('No new data. Skipping retrain.')
        return

    # 2. Apply feature engineering to new data
    df_live['request_time']  = pd.to_datetime(df_live['request_time'])
    df_live['hour']          = df_live['request_time'].dt.hour
    df_live['weekday']       = df_live['request_time'].dt.dayofweek
    df_live['month']         = df_live['request_time'].dt.month
    df_live['is_weekend']    = (df_live['weekday'] >= 5).astype(int)
    df_live['is_rush_hour']  = df_live['hour'].apply(
        lambda h: 1 if (7 <= h <= 9 or 17 <= h <= 19) else 0)
    df_live['time_period']   = df_live['hour'].apply(get_time_period)
    df_live['pickup_zone']   = df_live.apply(
        lambda r: coords_to_zone(r['pickup_lat'], r['pickup_lng']), axis=1)
    df_live['is_completed']  = (df_live['status'] == 'completed').astype(int)
    df_live['is_cancelled']  = (df_live['status'] == 'cancelled').astype(int)

    # 3. Merge with existing data
    combined = save_live_data_for_retraining(df_live)

    # 4. Retrain all models
    log = full_retrain_pipeline(combined)
    return log


print('✅ Retraining pipeline functions defined.')
print('To run manually: call nightly_retrain_with_supabase()')

---
## 🚀 STEP 8 — FastAPI Service Code

This generates the complete `main.py` file for your FastAPI AI microservice.

**Your Laravel backend calls this service** to get predictions.

In [ ]:
fastapi_code = '''
# RideConnect AI Service — main.py
# Run with: uvicorn main:app --host 0.0.0.0 --port 8001 --reload

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import joblib
import numpy as np
import json
import os
from tensorflow.keras.models import load_model

app = FastAPI(
    title="RideConnect AI Service",
    description="ML predictions for demand, matching, behavior and surge",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Load models on startup ────────────────────────────────────────────────────
MODELS_DIR = "./models"

demand_model    = load_model(f"{MODELS_DIR}/demand_lstm.h5")
matching_model  = joblib.load(f"{MODELS_DIR}/matching_rf.pkl")
behavior_model  = joblib.load(f"{MODELS_DIR}/behavior_gb.pkl")
surge_model     = joblib.load(f"{MODELS_DIR}/surge_xgb.pkl")
scaler_demand   = joblib.load(f"{MODELS_DIR}/scaler_demand.pkl")
le_zone         = joblib.load(f"{MODELS_DIR}/le_zone.pkl")
le_demand       = joblib.load(f"{MODELS_DIR}/le_demand.pkl")
le_period       = joblib.load(f"{MODELS_DIR}/le_period.pkl")

with open(f"{MODELS_DIR}/zone_mapping.json") as f:
    zone_mapping = json.load(f)

def encode_zone(zone: str) -> int:
    return zone_mapping.get(zone, 0)

def get_time_period(h: int) -> int:
    if   7  <= h <= 9:  period = "morning_rush"
    elif 17 <= h <= 19: period = "evening_rush"
    elif 20 <= h <= 23: period = "night"
    elif 0  <= h <= 5:  period = "late_night"
    else:               period = "off_peak"
    try:    return int(le_period.transform([period])[0])
    except: return 0


# ── Request schemas ───────────────────────────────────────────────────────────
class DemandRequest(BaseModel):
    hour: int
    weekday: int
    month: int
    is_weekend: int
    pickup_zone: str
    zone_hour_count: Optional[int] = 5

class MatchRequest(BaseModel):
    hour: int
    weekday: int
    is_weekend: int
    is_rush_hour: int
    pickup_zone: str
    dropoff_zone: str
    distance_km: float
    driver_rating: float
    driver_idle_time: float
    driver_cancel_rate: float
    driver_avg_rating: float
    driver_total_rides: int
    surge_multiplier: float
    demand_level: str

class BehaviorRequest(BaseModel):
    hour: int
    weekday: int
    is_rush_hour: int
    driver_rating: float
    driver_idle_time: float
    driver_total_rides: int
    driver_cancel_rate: float
    distance_km: float
    duration_min: float
    fare_rwf: float
    surge_multiplier: float
    pickup_zone: str

class SurgeRequest(BaseModel):
    hour: int
    weekday: int
    is_weekend: int
    is_rush_hour: int
    month: int
    pickup_zone: str
    distance_km: float
    zone_hour_count: Optional[int] = 5
    demand_level: str
    driver_idle_time: float
    wait_time_min: float


# ── Endpoints ─────────────────────────────────────────────────────────────────
@app.get("/health")
def health():
    return {"status": "running", "models": ["demand_lstm","matching_rf","behavior_gb","surge_xgb"]}


@app.post("/predict/demand")
def predict_demand(req: DemandRequest):
    zone_enc    = encode_zone(req.pickup_zone)
    period_enc  = get_time_period(req.hour)
    is_rush     = 1 if (7 <= req.hour <= 9 or 17 <= req.hour <= 19) else 0
    X = np.array([[req.hour, req.weekday, req.month, req.is_weekend,
                   is_rush, zone_enc, period_enc, req.zone_hour_count]])
    X_scaled = scaler_demand.transform(X).reshape(1, 1, 8)
    probs     = demand_model.predict(X_scaled)[0]
    pred_idx  = int(np.argmax(probs))
    label     = le_demand.inverse_transform([pred_idx])[0]
    return {
        "zone": req.pickup_zone,
        "predicted_demand": label,
        "confidence": round(float(np.max(probs)), 3),
        "probabilities": {le_demand.inverse_transform([i])[0]: round(float(p), 3)
                          for i, p in enumerate(probs)}
    }


@app.post("/predict/match")
def predict_match(req: MatchRequest):
    try:
        demand_enc = int(le_demand.transform([req.demand_level])[0])
    except:
        demand_enc = 1
    X = [[req.hour, req.weekday, req.is_weekend, req.is_rush_hour,
          encode_zone(req.pickup_zone), encode_zone(req.dropoff_zone),
          req.distance_km, req.driver_rating, req.driver_idle_time,
          req.driver_cancel_rate, req.driver_avg_rating, req.driver_total_rides,
          req.surge_multiplier, demand_enc]]
    prob = matching_model.predict_proba(X)[0][1]
    return {
        "completion_probability": round(float(prob), 3),
        "recommended": bool(prob >= 0.65)
    }


@app.post("/predict/behavior")
def predict_behavior(req: BehaviorRequest):
    X = [[req.hour, req.weekday, req.is_rush_hour,
          req.driver_rating, req.driver_idle_time,
          req.driver_total_rides, req.driver_cancel_rate,
          req.distance_km, req.duration_min, req.fare_rwf,
          req.surge_multiplier, encode_zone(req.pickup_zone)]]
    is_risky = bool(behavior_model.predict(X)[0])
    risk_prob = float(behavior_model.predict_proba(X)[0][1])
    return {
        "risky_driver": is_risky,
        "risk_score": round(risk_prob, 3),
        "risk_level": "high" if risk_prob > 0.7 else "medium" if risk_prob > 0.4 else "low"
    }


@app.post("/predict/surge")
def predict_surge(req: SurgeRequest):
    try:
        demand_enc = int(le_demand.transform([req.demand_level])[0])
    except:
        demand_enc = 1
    X = [[req.hour, req.weekday, req.is_weekend, req.is_rush_hour, req.month,
          encode_zone(req.pickup_zone), req.distance_km,
          req.zone_hour_count, demand_enc,
          req.driver_idle_time, req.wait_time_min]]
    multiplier = float(surge_model.predict(X)[0])
    multiplier = round(max(1.0, min(2.5, multiplier)), 2)  # clamp to valid range
    return {"surge_multiplier": multiplier}


@app.get("/models/info")
def models_info():
    log_path = f"{MODELS_DIR}/retrain_log.json"
    if os.path.exists(log_path):
        with open(log_path) as f:
            return json.load(f)
    return {"message": "No retrain log yet. Run initial training first."}
'''

with open('/content/rideconnect_data/main.py', 'w') as f:
    f.write(fastapi_code)

print('✅ FastAPI service code written to: /content/rideconnect_data/main.py')

---
## 📦 STEP 9 — Export Everything & Download

Download all trained models + FastAPI code as a ZIP file to deploy on your server

In [ ]:
import shutil

# Copy FastAPI file into models folder
shutil.copy('/content/rideconnect_data/main.py', '/content/rideconnect_models/main.py')

# Create requirements.txt
requirements = """fastapi==0.110.0
uvicorn==0.29.0
pandas==2.2.0
numpy==1.26.0
scikit-learn==1.4.0
xgboost==2.0.3
tensorflow==2.15.0
joblib==1.3.2
requests==2.31.0
supabase==2.4.0
python-dotenv==1.0.1
"""
with open('/content/rideconnect_models/requirements.txt', 'w') as f:
    f.write(requirements)

# Create README
readme = """# RideConnect AI Service

## Setup
```bash
pip install -r requirements.txt
uvicorn main:app --host 0.0.0.0 --port 8001
```

## Endpoints
- POST /predict/demand   → Demand level for a zone at a given time
- POST /predict/match    → Completion probability for a driver-passenger pair
- POST /predict/behavior → Driver risk assessment
- POST /predict/surge    → Surge price multiplier
- GET  /health           → Service health check
- GET  /models/info      → Last retrain stats

## Laravel Integration
Call this service from your Laravel controllers using Http::post().
See full integration guide in the notebook comments.
"""
with open('/content/rideconnect_models/README.md', 'w') as f:
    f.write(readme)

# Zip everything
shutil.make_archive('/content/rideconnect_ai_service', 'zip', '/content/rideconnect_models')

print('✅ Package ready!')
print('Files included:')
for f in sorted(os.listdir('/content/rideconnect_models')):
    size = os.path.getsize(f'/content/rideconnect_models/{f}')
    print(f'  {f:40s} {size/1024:.1f} KB')

# Download the ZIP
from google.colab import files
files.download('/content/rideconnect_ai_service.zip')
print('\n📦 Downloading rideconnect_ai_service.zip...')

---
## 📋 STEP 10 — Laravel Integration Reference

Copy-paste these snippets into your Laravel controllers

In [ ]:
laravel_code = '''
<?php
// ============================================================
// RideConnect — Laravel AI Service Integration
// Add AI_SERVICE_URL=http://localhost:8001 to your .env
// ============================================================

// ── 1. app/Services/AiService.php ────────────────────────────
namespace App\\Services;
use Illuminate\\Support\\Facades\\Http;

class AiService {
    private string $base;
    public function __construct() {
        $this->base = config(\'services.ai.url\', env(\'AI_SERVICE_URL\', \'http://localhost:8001\'));
    }

    /** Predict demand level for a zone at current time */
    public function predictDemand(string $zone): array {
        $now = now();
        return Http::post("{$this->base}/predict/demand", [
            \'hour\'            => $now->hour,
            \'weekday\'         => $now->dayOfWeek,
            \'month\'           => $now->month,
            \'is_weekend\'      => $now->isWeekend() ? 1 : 0,
            \'pickup_zone\'     => $zone,
            \'zone_hour_count\' => 5,  // pass real count from your DB
        ])->json();
    }

    /** Score a driver-passenger match — returns completion probability */
    public function scoreMatch(array $ride, array $driver): float {
        $now = now();
        $resp = Http::post("{$this->base}/predict/match", [
            \'hour\'               => $now->hour,
            \'weekday\'            => $now->dayOfWeek,
            \'is_weekend\'         => $now->isWeekend() ? 1 : 0,
            \'is_rush_hour\'       => in_array($now->hour, [7,8,9,17,18,19]) ? 1 : 0,
            \'pickup_zone\'        => $ride[\'pickup_zone\'],
            \'dropoff_zone\'       => $ride[\'dropoff_zone\'],
            \'distance_km\'        => $ride[\'distance_km\'],
            \'driver_rating\'      => $driver[\'rating\'],
            \'driver_idle_time\'   => $driver[\'idle_time\'],
            \'driver_cancel_rate\' => $driver[\'cancel_rate\'],
            \'driver_avg_rating\'  => $driver[\'avg_rating\'],
            \'driver_total_rides\' => $driver[\'total_rides\'],
            \'surge_multiplier\'   => 1.0,
            \'demand_level\'       => \'medium\',
        ])->json()[\'completion_probability\'] ?? 0.5;
        return $resp;
    }

    /** Check if a driver is risky before assigning ride */
    public function checkDriverBehavior(array $driver, array $ride): array {
        $now = now();
        return Http::post("{$this->base}/predict/behavior", [
            \'hour\'              => $now->hour,
            \'weekday\'           => $now->dayOfWeek,
            \'is_rush_hour\'      => in_array($now->hour,[7,8,9,17,18,19]) ? 1 : 0,
            \'driver_rating\'     => $driver[\'rating\'],
            \'driver_idle_time\'  => $driver[\'idle_time\'],
            \'driver_total_rides\'=> $driver[\'total_rides\'],
            \'driver_cancel_rate\'=> $driver[\'cancel_rate\'],
            \'distance_km\'       => $ride[\'distance_km\'],
            \'duration_min\'      => $ride[\'estimated_duration\'],
            \'fare_rwf\'          => $ride[\'fare\'],
            \'surge_multiplier\'  => 1.0,
            \'pickup_zone\'       => $ride[\'pickup_zone\'],
        ])->json();
    }

    /** Get surge multiplier for fare calculation */
    public function getSurgeMultiplier(string $zone, float $distance, float $waitTime): float {
        $now = now();
        return Http::post("{$this->base}/predict/surge", [
            \'hour\'            => $now->hour,
            \'weekday\'         => $now->dayOfWeek,
            \'is_weekend\'      => $now->isWeekend() ? 1 : 0,
            \'is_rush_hour\'    => in_array($now->hour,[7,8,9,17,18,19]) ? 1 : 0,
            \'month\'           => $now->month,
            \'pickup_zone\'     => $zone,
            \'distance_km\'     => $distance,
            \'zone_hour_count\' => 5,
            \'demand_level\'    => \'medium\',
            \'driver_idle_time\'=> 10,
            \'wait_time_min\'   => $waitTime,
        ])->json()[\'surge_multiplier\'] ?? 1.0;
    }
}

// ── 2. In RideController@store ────────────────────────────────
// $ai = new AiService();
// $demand   = $ai->predictDemand($ride->pickup_zone);
// $surge    = $ai->getSurgeMultiplier($zone, $distance, $waitTime);
// $bestDriver = collect($nearbyDrivers)
//     ->sortByDesc(fn($d) => $ai->scoreMatch($rideData, $d))
//     ->first();
'''

with open('/content/rideconnect_data/laravel_integration.php', 'w') as f:
    f.write(laravel_code)

print('✅ Laravel integration code saved to laravel_integration.php')
print()
print('=== HOW IT ALL CONNECTS ===')
print()
print('  kigali_rides.csv')
print('       ↓ (this notebook)')
print('  Feature Engineering → 4 Trained Models')
print('       ↓')
print('  FastAPI service (port 8001)')
print('       ↓  ↑ (HTTP calls)')
print('  Laravel Backend (your existing API)')
print('       ↑')
print('  Supabase DB → nightly CSV export → retrain pipeline')
print('       ↑')
print('  Flutter App (passenger + driver requests)')

---
## ✅ Done!

### What you now have:

| File | Purpose |
|------|------|
| `demand_lstm.h5` | Predicts demand level (low/medium/high) per zone |
| `matching_rf.pkl` | Scores driver-passenger matches |
| `behavior_gb.pkl` | Flags risky drivers |
| `surge_xgb.pkl` | Predicts fare surge multiplier |
| `main.py` | FastAPI server exposing all 4 models |
| `laravel_integration.php` | Copy-paste service class for your Laravel app |

### Next steps:
1. **Download** the ZIP from Step 9
2. **Deploy** the FastAPI service on your server (or locally during dev)
3. **Fill in** your Supabase credentials in Step 6
4. **Schedule** the retrain cell to run nightly as more rides accumulate
5. **Add** `AiService.php` to your Laravel app and call it from your controllers